In [1]:
# Exercise 

import os
import json
from openai import OpenAI
import sqlite3
import gradio as gr
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    print(f"API key loaded successfully and starts with {api_key[:8]}")
else:
    print("API key not found")

openai = OpenAI(api_key=api_key)
model = "gpt-4.1-mini"

sys_msg = """
    You are a helpful assistant giving me pricing of books for a customer
    Answer in one line for the question asked and be precise
    """

#book_prices = {"Maths" : 500 , "Science": 600 , "Social" : 700}

price_function = {
    "name" : "get_book_price",
    "description" : "Get the price of the book",
    "parameters":{
        "type" : "object",
        "properties" : {
           "book": {
            "type" : "string",
            "description" : "The price of the book",
           },
        },
         "required" : ["book"],
        "additionalParameters" : False,
    }
}

set_price_function = {
    "name" : "set_book_price",
    "description" : "Set the price of the book",
    "parameters":{
        "type" : "object",
        "properties" : {
           "book": {
            "type" : "string",
            "description" : "The price of the book",
           },
           "price": {
            "type" : "number",
            "description" : "The new ticket price",
           },
        },
         "required" : ["book","price"],
        "additionalParameters" : False,
    }
}


tools = [
    {"type": "function", "function": price_function},
    {"type": "function", "function": set_price_function},
]

tools




API key loaded successfully and starts with sk-proj-


[{'type': 'function',
  'function': {'name': 'get_book_price',
   'description': 'Get the price of the book',
   'parameters': {'type': 'object',
    'properties': {'book': {'type': 'string',
      'description': 'The price of the book'}},
    'required': ['book'],
    'additionalParameters': False}}},
 {'type': 'function',
  'function': {'name': 'set_book_price',
   'description': 'Set the price of the book',
   'parameters': {'type': 'object',
    'properties': {'book': {'type': 'string',
      'description': 'The price of the book'},
     'price': {'type': 'number', 'description': 'The new ticket price'}},
    'required': ['book', 'price'],
    'additionalParameters': False}}}]

In [ ]:
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (book TEXT PRIMARY KEY, price REAL)')
    conn.commit()


def get_book_price(book):
    print(f"Tool called for book price : {book}")
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price from prices where book = ?', (book.lower(),))
        result = cursor.fetchone()
        return f"The {book} book price is ${result[0]}" if result else "No price data available for this book"
        


def set_book_price(book,price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (book, price) VALUES (?, ?) ON CONFLICT(book) DO UPDATE SET price = ?', (book.lower(), price, price))
        conn.commit()

def chat(message,history):
    history = [{"role" : h["role"], "content" : h["content"]} for h in history]
    messages = [{"role": "system", "content": sys_msg}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model = model,messages = messages, tools = tools)

    if response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)

    response = openai.chat.completions.create(model = model,messages = messages, tools = tools)
    return response.choices[0].message.content

def handle_tool_calls(message):
    def run_get_book_price(arguments):
        book = arguments.get('book')
        return get_book_price(book)


    def run_set_book_price(arguments):
        book = arguments.get('book')
        price = arguments.get('price')
        set_book_price(book,price)
        return f"Updated book price for {book} to ${price}" 

    tool_handlers = {
        "get_book_price": run_get_book_price,
        "set_book_price": run_set_book_price,
    }

    responses = []
    for tool_call in message.tool_calls:
        arguments = json.loads(tool_call.function.arguments)
        handler = tool_handlers.get(tool_call.function.name)
        if handler is None:
            continue
        result  = handler(arguments)
        responses.append({
            "role": "tool",
            "content": result,
            "tool_call_id": tool_call.id,
        })
    return responses


book_prices = {"Maths": 500, "Science": 600, "Social": 700}
for book, price in book_prices.items():
    set_book_price(book, price)

get_book_price("Maths")

Tool called for book price : Maths


'The Maths book price is $500.0'

In [5]:
gr.ChatInterface(fn=chat, type="messages").launch()


* Running on local URL:  http://127.0.0.1:7874
* To create a public link, set `share=True` in `launch()`.
